<a href="https://www.kaggle.com/code/whatismy/vit-gauss?scriptVersionId=282272595" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install gpytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.9/279.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViTModel, ViTImageProcessor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import gpytorch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import torch.nn as nn
import warnings
import gc
import json
from datetime import datetime

2025-11-27 19:19:29.604196: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764271169.785944      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764271169.836188      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ==================== Dataset ====================
class EuroSATDataset(Dataset):
    def __init__(self, filenames, labels, image_dir, processor):
        self.filenames = filenames
        self.labels = labels
        self.image_dir = image_dir
        self.processor = processor
        
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img_path = self.image_dir + self.filenames[idx]
        label = int(self.labels[idx])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        inputs = self.processor(images=image, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0), label

# ==================== Feature Extraction ====================
def extract_features(dataloader, vit_model, device):
    vit_model.eval()
    features, labels = [], []
    
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting features"):
            imgs = imgs.to(device)
            outputs = vit_model(pixel_values=imgs)
            features.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
            labels.append(lbls.numpy())
    
    return np.vstack(features), np.concatenate(labels)

# ==================== OPTIMIZED Binary GP Model ====================
class OptimizedBinaryGP(gpytorch.models.ApproximateGP):
    """
    Optimized Binary GP with:
    - Better kernel initialization
    - ARD for automatic feature selection
    - Adaptive inducing points
    """
    def __init__(self, train_x, train_y, num_inducing=100, use_ard=False):
        # Smart inducing point initialization: sample from both classes
        pos_indices = torch.where(train_y == 1)[0]
        neg_indices = torch.where(train_y == 0)[0]
        
        # Split inducing points between classes
        num_pos = min(num_inducing // 2, len(pos_indices))
        num_neg = num_inducing - num_pos
        
        if len(pos_indices) > 0 and len(neg_indices) > 0:
            pos_sample = pos_indices[torch.randperm(len(pos_indices))[:num_pos]]
            neg_sample = neg_indices[torch.randperm(len(neg_indices))[:num_neg]]
            inducing_idx = torch.cat([pos_sample, neg_sample])
        else:
            inducing_idx = torch.randperm(train_x.size(0))[:num_inducing]
        
        inducing_points = train_x[inducing_idx]
        
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=inducing_points.size(0)
        )
        
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True
        )
        
        super().__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        
        # LINEAR kernel - works best for pre-trained ViT features
        # Add constant kernel for numerical stability
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.LinearKernel() + 
            gpytorch.kernels.ConstantKernel()
        )
        
        # Good initialization
        self.covar_module.outputscale = 1.0
        
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# ==================== OPTIMIZED Training ====================
def train_binary_gp_optimized(model, likelihood, train_x, train_y, 
                               n_epochs=200, initial_lr=0.1):
    """
    Optimized training with:
    - Adaptive learning rate
    - Better convergence criteria
    - Regularization
    """
    model.train()
    likelihood.train()
    
    # Separate optimizers for different parameter groups
    optimizer = torch.optim.Adam([
        {'params': model.variational_parameters(), 'lr': initial_lr},
        {'params': model.hyperparameters(), 'lr': initial_lr * 0.1},
        {'params': likelihood.parameters(), 'lr': initial_lr * 0.1},
    ])
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15, verbose=False
    )
    
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
    
    best_loss = float('inf')
    patience = 0
    max_patience = 40
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        loss_val = loss.item()
        scheduler.step(loss_val)
        
        if loss_val < best_loss:
            best_loss = loss_val
            patience = 0
        else:
            patience += 1
        
        if patience >= max_patience:
            break
    
    return model, likelihood, best_loss

# ==================== OPTIMIZED One-vs-Rest Classifier ====================
class OptimizedOneVsRestGP:
    """
    Optimized One-vs-Rest GP with:
    - Smart inducing point initialization
    - Linear kernel for pre-trained features
    - Calibrated predictions
    """
    def __init__(self, num_classes, num_inducing=120):
        self.num_classes = num_classes
        self.num_inducing = num_inducing
        self.models = []
        self.likelihoods = []
        self.class_priors = []
    
    def fit(self, train_x, train_y, n_epochs=200, initial_lr=0.12):
        """Train optimized binary GPs"""
        
        
        for class_idx in range(self.num_classes):
            
            # Binary labels
            train_y_binary = (train_y == class_idx).float()
            
            num_pos = train_y_binary.sum().item()
            num_neg = (train_y_binary == 0).sum().item()
            
            
            # Store class prior
            self.class_priors.append(num_pos / len(train_y))
            
            # Create optimized GP
            model = OptimizedBinaryGP(
                train_x, train_y_binary, 
                num_inducing=self.num_inducing
            ).to(train_x.device)
            
            likelihood = gpytorch.likelihoods.BernoulliLikelihood().to(train_x.device)
            
            # Train with optimization
            model, likelihood, final_loss = train_binary_gp_optimized(
                model, likelihood, train_x, train_y_binary,
                n_epochs=n_epochs, initial_lr=initial_lr
            )
            
            
            print(f"Final loss: {final_loss:.4f}")
            
            self.models.append(model)
            self.likelihoods.append(likelihood)
        
    
    def predict(self, test_x, calibrate=True, batch_size=100):
        """
        Predict with optional calibration
        """
        print("\nMaking predictions...")
        
        for model, likelihood in zip(self.models, self.likelihoods):
            model.eval()
            likelihood.eval()
        
        all_scores = []
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            for class_idx in tqdm(range(self.num_classes), desc="Computing scores"):
                model = self.models[class_idx]
                likelihood = self.likelihoods[class_idx]
                
                class_scores = []
                
                for i in range(0, test_x.size(0), batch_size):
                    batch_x = test_x[i:min(i+batch_size, test_x.size(0))]
                    
                    output = model(batch_x)
                    pred = likelihood(output)
                    
                    # Get probability of positive class
                    scores = pred.mean
                    class_scores.append(scores.cpu())
                
                class_scores = torch.cat(class_scores)
                all_scores.append(class_scores)
        
        # Stack scores: (num_classes, num_samples)
        all_scores = torch.stack(all_scores)
        
        if calibrate:
            # Apply prior calibration (helps with imbalanced classes)
            priors = torch.tensor(self.class_priors).unsqueeze(1)
            all_scores = all_scores * priors
        
        # Choose class with highest score
        predictions = all_scores.argmax(dim=0).numpy()
        
        return predictions, all_scores.numpy()

# ==================== Main ====================
def main():
    print("="*70)
    print("OPTIMIZED Gaussian Process Classification")
    print("="*70)
    
    # Load data
    print("\nLoading data...")
    train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')
    test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')
    
    # Setup labels
    if 'ClassName' in train_df.columns:
        classes = sorted(train_df['ClassName'].unique())
        train_df['label_idx'] = train_df['ClassName'].map({c: i for i, c in enumerate(classes)})
        test_df['label_idx'] = test_df['ClassName'].map({c: i for i, c in enumerate(classes)}) if 'ClassName' in test_df.columns else test_df['Label']
    else:
        train_df['label_idx'] = train_df['Label']
        test_df['label_idx'] = test_df['Label']
    
    num_classes = len(train_df['label_idx'].unique())
    print(f"Number of classes: {num_classes}")
    
    # Subset (20 per class)
    train_subset = pd.concat([
        train_df[train_df['label_idx'] == i].sample(n=20, random_state=42)
        for i in range(num_classes)
    ], ignore_index=True)
    
    print(f"Training samples: {len(train_subset)} ({len(train_subset)//num_classes} per class)")
    print(f"Test samples: {len(test_df)}")
    
    # Extract arrays
    train_files = train_subset['Filename'].values
    train_labels = train_subset['label_idx'].values
    test_files = test_df['Filename'].values
    test_labels = test_df['label_idx'].values
    
    image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'
    
    # Load ViT
    print("\nLoading ViT...")
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    
    import transformers
    transformers.logging.set_verbosity_error()
    vit = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)
    transformers.logging.set_verbosity_warning()
    
    for p in vit.parameters():
        p.requires_grad = False
    
    # Datasets
    train_ds = EuroSATDataset(train_files, train_labels, image_dir, processor)
    test_ds = EuroSATDataset(test_files, test_labels, image_dir, processor)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    
    # Extract features
    print("\n" + "="*70)
    print("Feature Extraction")
    print("="*70)
    train_feat, train_lbl = extract_features(train_loader, vit, device)
    test_feat, test_lbl = extract_features(test_loader, vit, device)
    

    
    print(f"Extracted: Train {train_feat.shape}, Test {test_feat.shape}")
    
    # Preprocess
    print("\nPreprocessing...")
    scaler = StandardScaler()
    train_feat = scaler.fit_transform(train_feat)
    test_feat = scaler.transform(test_feat)
    
    # OPTIMIZATION: Use more PCA components for better feature retention
    pca_components = 128  # Increased from 100
    pca = PCA(n_components=pca_components, random_state=42)
    train_feat = pca.fit_transform(train_feat)
    test_feat = pca.transform(test_feat)
    
    print(f"PCA: {pca_components} components, {pca.explained_variance_ratio_.sum():.4f} variance")
    
    # Convert to tensors
    train_x = torch.tensor(train_feat, dtype=torch.float32).to(device)
    train_y = torch.tensor(train_lbl, dtype=torch.long).to(device)
    test_x = torch.tensor(test_feat, dtype=torch.float32).to(device)
    
    # Train OPTIMIZED GP
    print("\n" + "="*70)
    print("Training OPTIMIZED GP")
    print("="*70)
    
    gp_classifier = OptimizedOneVsRestGP(
        num_classes=num_classes,
        num_inducing=130  # Slightly increased
    )
    
    gp_classifier.fit(
        train_x, train_y, 
        n_epochs=2000,  # More epochs
        initial_lr=0.12,  # Higher initial LR
    )
    
    # Predict with calibration
    print("\n" + "="*70)
    print("Evaluation")
    print("="*70)
    
    predictions, scores = gp_classifier.predict(
        test_x, 
        calibrate=True,  # Use prior calibration
        batch_size=100
    )
    
    # Calculate accuracy
    accuracy = accuracy_score(test_lbl, predictions)
    
    print(f"\n{'='*70}")
    print(f"OPTIMIZED GP TEST ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"{'='*70}")
    
    # Detailed report
    print("\nClassification Report:")
    print("="*70)
    print(classification_report(test_lbl, predictions,
                                target_names=[f"Class {i}" for i in range(num_classes)],
                                digits=4))
    
    # Per-class performance
    print("\nPer-Class Performance:")
    print("-"*70)
    for i in range(num_classes):
        mask = test_lbl == i
        if mask.sum() > 0:
            acc = accuracy_score(test_lbl[mask], predictions[mask])
            avg_score = scores[i, mask].mean()
            std_score = scores[i, mask].std()
            print(f"Class {i:2d}: Acc={acc*100:6.2f}% | "
                  f"Score={avg_score:.3f}±{std_score:.3f} | "
                  f"N={mask.sum():4d}")
    
    print("\n" + "="*70)
    print("OPTIMIZATION FEATURES USED:")
    print("✓ Linear + Constant Kernel (optimal for ViT features)")
    print("✓ Class-balanced inducing point initialization")
    print("✓ Adaptive learning rate with ReduceLROnPlateau")
    print("✓ 128 PCA components (more information retained)")
    print("✓ Prior calibration in prediction")
    print("✓ Extended training (200 epochs with early stopping)")
    print("="*70)
    
    return accuracy

if __name__ == "__main__":
    main()

Using device: cuda
GPU: Tesla T4
OPTIMIZED Gaussian Process Classification

Loading data...
Number of classes: 10
Training samples: 200 (20 per class)
Test samples: 2700

Loading ViT...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


Feature Extraction


Extracting features: 100%|██████████| 85/85 [00:50<00:00,  1.69it/s]


Extracted: Train (200, 768), Test (2700, 768)

Preprocessing...
PCA: 128 components, 0.9549 variance

Training OPTIMIZED GP
Final loss: 0.1845
Final loss: 0.1410
Final loss: 0.1773
Final loss: 0.2343
Final loss: 0.1625
Final loss: 0.1939
Final loss: 0.2484
Final loss: 0.1694
Final loss: 0.2454
Final loss: 0.1488

Evaluation

Making predictions...


Computing scores: 100%|██████████| 10/10 [00:01<00:00,  8.05it/s]


OPTIMIZED GP TEST ACCURACY: 0.8619 (86.19%)

Classification Report:
              precision    recall  f1-score   support

     Class 0     0.8700    0.9367    0.9021       300
     Class 1     0.9148    0.9667    0.9400       300
     Class 2     0.8253    0.9133    0.8671       300
     Class 3     0.7437    0.7080    0.7254       250
     Class 4     0.8662    0.9320    0.8979       250
     Class 5     0.8037    0.8800    0.8401       200
     Class 6     0.8889    0.7040    0.7857       250
     Class 7     0.9298    0.9267    0.9282       300
     Class 8     0.8107    0.6680    0.7325       250
     Class 9     0.9197    0.9167    0.9182       300

    accuracy                         0.8619      2700
   macro avg     0.8573    0.8552    0.8537      2700
weighted avg     0.8615    0.8619    0.8593      2700


Per-Class Performance:
----------------------------------------------------------------------
Class  0: Acc= 93.67% | Score=0.064±0.026 | N= 300
Class  1: Acc= 96.67% | Sc

In [4]:
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== Dataset ====================
class EuroSATDataset(Dataset):
    def __init__(self, filenames, labels, image_dir, processor, augment=False, hard_classes=None):
        self.filenames = filenames
        self.labels = labels
        self.image_dir = image_dir
        self.processor = processor
        self.augment = augment
        self.hard_classes = hard_classes if hard_classes is not None else []
        
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img_path = self.image_dir + self.filenames[idx]
        label = int(self.labels[idx])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.augment:
            import torchvision.transforms as T
            
            if label in self.hard_classes:
                augment_transforms = T.Compose([
                    T.RandomHorizontalFlip(p=0.5),
                    T.RandomVerticalFlip(p=0.5),
                    T.RandomRotation(degrees=180),
                    T.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15)),
                    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
                    T.RandomResizedCrop(size=224, scale=(0.75, 1.0)),
                ])
            else:
                augment_transforms = T.Compose([
                    T.RandomHorizontalFlip(p=0.5),
                    T.RandomVerticalFlip(p=0.5),
                    T.RandomRotation(degrees=90),
                    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                ])
            
            image = augment_transforms(image)
        
        inputs = self.processor(images=image, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0), label

# ==================== ViT with Last 2 Layers Unfrozen ====================
class FineTunedViT(nn.Module):
    def __init__(self, num_layers_to_unfreeze=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        
        for param in self.vit.parameters():
            param.requires_grad = False
        
        total_layers = len(self.vit.encoder.layer)
        for i in range(total_layers - num_layers_to_unfreeze, total_layers):
            for param in self.vit.encoder.layer[i].parameters():
                param.requires_grad = True
    
    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        return outputs.last_hidden_state[:, 0, :]

# ==================== Binary GP ====================
class BinaryGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, train_x, train_y, num_inducing=100):
        pos_indices = torch.where(train_y == 1)[0]
        neg_indices = torch.where(train_y == 0)[0]
        
        num_pos = min(num_inducing // 2, len(pos_indices))
        num_neg = num_inducing - num_pos
        
        if len(pos_indices) > 0 and len(neg_indices) > 0:
            pos_sample = pos_indices[torch.randperm(len(pos_indices))[:num_pos]]
            neg_sample = neg_indices[torch.randperm(len(neg_indices))[:num_neg]]
            inducing_idx = torch.cat([pos_sample, neg_sample])
        else:
            inducing_idx = torch.randperm(train_x.size(0))[:num_inducing]
        
        inducing_points = train_x[inducing_idx].detach().clone()
        
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=inducing_points.size(0)
        )
        
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution,
            learn_inducing_locations=True
        )
        
        super().__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.LinearKernel() + 
            gpytorch.kernels.ConstantKernel()
        )
        
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# ==================== Training Functions ====================
def finetune_vit_layers(vit_model, train_loader, n_epochs=40, lr=2e-5, verbose=False):
    num_classes = 10
    classifier = nn.Linear(768, num_classes).to(device)
    
    vit_model.train()
    classifier.train()
    
    class_weights = torch.ones(num_classes).to(device)
    class_weights[3] = 2.0
    class_weights[6] = 2.0
    class_weights[8] = 2.0
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    optimizer = torch.optim.AdamW([
        {'params': filter(lambda p: p.requires_grad, vit_model.parameters()), 
         'lr': lr, 'weight_decay': 0.01},
        {'params': classifier.parameters(), 'lr': lr * 10, 'weight_decay': 0.01}
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-7
    )
    
    best_loss = float('inf')
    patience = 0
    max_patience = 15
    
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            features = vit_model(imgs)
            logits = classifier(features)
            loss = criterion(logits, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        acc = 100. * correct / total
        avg_loss = epoch_loss / len(train_loader)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience = 0
        else:
            patience += 1
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs} - Loss: {avg_loss:.4f} - Acc: {acc:.2f}%")
        
        if patience >= max_patience:
            break
    
    del classifier
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return vit_model

def train_gps(vit_model, train_loader, train_labels, num_classes, verbose=False):
    vit_model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            features = vit_model(imgs)
            all_features.append(features.cpu())
            all_labels.append(labels)
    
    train_features = torch.cat(all_features).to(device)
    train_y = torch.cat(all_labels).to(device)
    
    scaler = StandardScaler()
    train_feat_np = scaler.fit_transform(train_features.cpu().numpy())
    train_features = torch.tensor(train_feat_np, dtype=torch.float32).to(device)
    
    models = []
    likelihoods = []
    
    for class_idx in range(num_classes):
        binary_labels = (train_y == class_idx).float()
        
        model = BinaryGPModel(train_features, binary_labels, num_inducing=100).to(device)
        likelihood = gpytorch.likelihoods.BernoulliLikelihood().to(device)
        
        model.train()
        likelihood.train()
        
        optimizer = torch.optim.Adam([
            {'params': model.variational_parameters(), 'lr': 0.05},
            {'params': model.hyperparameters(), 'lr': 0.005},
            {'params': likelihood.parameters(), 'lr': 0.005},
        ])
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=15
        )
        
        mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
        
        best_loss = float('inf')
        patience = 0
        
        for epoch in range(150):
            optimizer.zero_grad()
            output = model(train_features)
            loss = -mll(output, binary_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step(loss.item())
            
            if loss.item() < best_loss:
                best_loss = loss.item()
                patience = 0
            else:
                patience += 1
            
            if patience >= 30:
                break
        
        models.append(model)
        likelihoods.append(likelihood)
    
    return models, likelihoods, scaler

def predict(vit_model, models, likelihoods, test_loader, scaler, num_classes):
    vit_model.eval()
    for model, likelihood in zip(models, likelihoods):
        model.eval()
        likelihood.eval()
    
    all_preds = []
    
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        for imgs, _ in test_loader:
            imgs = imgs.to(device)
            features = vit_model(imgs).cpu().numpy()
            features = scaler.transform(features)
            features = torch.tensor(features, dtype=torch.float32).to(device)
            
            class_scores = []
            for class_idx in range(num_classes):
                output = models[class_idx](features)
                pred = likelihoods[class_idx](output)
                scores = pred.mean
                class_scores.append(scores)
            
            class_scores = torch.stack(class_scores, dim=0)
            batch_preds = class_scores.argmax(dim=0)
            all_preds.append(batch_preds.cpu())
    
    predictions = torch.cat(all_preds).numpy()
    return predictions

# ==================== Single Experiment ====================
def run_single_experiment(train_df, test_df, image_dir, processor, 
                          samples_per_class, seed, num_classes=10):
    """Run one experiment with given parameters"""
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Clear GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    # Sample training data
    train_subset = pd.concat([
        train_df[train_df['label_idx'] == i].sample(n=min(samples_per_class, len(train_df[train_df['label_idx'] == i])), 
                                                     random_state=seed)
        for i in range(num_classes)
    ], ignore_index=True)
    
    train_files = train_subset['Filename'].values
    train_labels = train_subset['label_idx'].values
    test_files = test_df['Filename'].values
    test_labels = test_df['label_idx'].values
    
    # Datasets
    hard_classes = [3, 6, 8]
    train_ds = EuroSATDataset(train_files, train_labels, image_dir, processor, 
                               augment=True, hard_classes=hard_classes)
    train_ds_no_aug = EuroSATDataset(train_files, train_labels, image_dir, processor, 
                                      augment=False)
    test_ds = EuroSATDataset(test_files, test_labels, image_dir, processor, augment=False)
    
    train_loader_aug = DataLoader(train_ds, batch_size=min(16, len(train_ds)), shuffle=True)
    train_loader_no_aug = DataLoader(train_ds_no_aug, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    
    # Create model
    import transformers
    transformers.logging.set_verbosity_error()
    
    vit_model = FineTunedViT(num_layers_to_unfreeze=2).to(device)
    
    # Train
    vit_model = finetune_vit_layers(vit_model, train_loader_aug, n_epochs=40, lr=2e-5, verbose=False)
    models, likelihoods, scaler = train_gps(vit_model, train_loader_no_aug, train_labels, num_classes, verbose=False)
    
    # Predict
    predictions = predict(vit_model, models, likelihoods, test_loader, scaler, num_classes)
    accuracy = accuracy_score(test_labels, predictions)
    
    # Per-class accuracy
    per_class_acc = {}
    for i in range(num_classes):
        mask = test_labels == i
        if mask.sum() > 0:
            per_class_acc[i] = accuracy_score(test_labels[mask], predictions[mask])
    
    # Cleanup
    del vit_model, models, likelihoods
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    return accuracy, per_class_acc

# ==================== Main Experiments ====================
def main():
    print("="*70)
    print("GP COMPREHENSIVE EXPERIMENTS")
    print("="*70)
    
    # Load data
    print("\nLoading data...")
    train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')
    test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')
    
    if 'ClassName' in train_df.columns:
        classes = sorted(train_df['ClassName'].unique())
        train_df['label_idx'] = train_df['ClassName'].map({c: i for i, c in enumerate(classes)})
        test_df['label_idx'] = test_df['ClassName'].map({c: i for i, c in enumerate(classes)}) if 'ClassName' in test_df.columns else test_df['Label']
    else:
        train_df['label_idx'] = train_df['Label']
        test_df['label_idx'] = test_df['Label']
    
    num_classes = len(train_df['label_idx'].unique())
    image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    
    # Results storage
    results = {
        'data_size_experiments': {},
        'seed_experiments': {}
    }
    
    # EXPERIMENT 1: Vary data size (samples per class: 100, 50, 20, 10, 5, 1)
    print("\n" + "="*70)
    print("EXPERIMENT 1: Varying Data Size")
    print("="*70)
    
    data_sizes = [100, 50, 20, 10, 5, 1]
    seed = 42
    
    for samples_per_class in data_sizes:
        print(f"\n--- Running with {samples_per_class} samples per class (seed={seed}) ---")
        
        accuracy, per_class_acc = run_single_experiment(
            train_df, test_df, image_dir, processor, 
            samples_per_class, seed, num_classes
        )
        
        results['data_size_experiments'][samples_per_class] = {
            'accuracy': float(accuracy),
            'per_class': {int(k): float(v) for k, v in per_class_acc.items()},
            'seed': seed
        }
        
        print(f"Accuracy: {accuracy*100:.2f}%")
        print(f"Per-class (hard): Class 3: {per_class_acc.get(3, 0)*100:.2f}%, "
              f"Class 6: {per_class_acc.get(6, 0)*100:.2f}%, "
              f"Class 8: {per_class_acc.get(8, 0)*100:.2f}%")
    
    # EXPERIMENT 2: Vary seed (20 samples per class, seeds: 42, 123, 456, 789, 2024)
    print("\n" + "="*70)
    print("EXPERIMENT 2: Varying Seed (20 samples/class)")
    print("="*70)
    
    samples_per_class = 20
    seeds = [42, 123, 456, 789, 2024]
    
    for seed in seeds:
        print(f"\n--- Running with seed={seed} (20 samples per class) ---")
        
        accuracy, per_class_acc = run_single_experiment(
            train_df, test_df, image_dir, processor, 
            samples_per_class, seed, num_classes
        )
        
        results['seed_experiments'][seed] = {
            'accuracy': float(accuracy),
            'per_class': {int(k): float(v) for k, v in per_class_acc.items()},
            'samples_per_class': samples_per_class
        }
        
        print(f"Accuracy: {accuracy*100:.2f}%")
        print(f"Per-class (hard): Class 3: {per_class_acc.get(3, 0)*100:.2f}%, "
              f"Class 6: {per_class_acc.get(6, 0)*100:.2f}%, "
              f"Class 8: {per_class_acc.get(8, 0)*100:.2f}%")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f'/kaggle/working/gp_experiments_{timestamp}.json'
    
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print("\n" + "="*70)
    print("FINAL RESULTS SUMMARY")
    print("="*70)
    
    print("\n1. DATA SIZE VARIATION (seed=42):")
    print("-" * 70)
    print(f"{'Samples/Class':<15} {'Accuracy':<12} {'Class 3':<10} {'Class 6':<10} {'Class 8':<10}")
    print("-" * 70)
    for size in data_sizes:
        res = results['data_size_experiments'][size]
        print(f"{size:<15} {res['accuracy']*100:>10.2f}% {res['per_class'].get(3, 0)*100:>8.2f}% "
              f"{res['per_class'].get(6, 0)*100:>8.2f}% {res['per_class'].get(8, 0)*100:>8.2f}%")
    
    print("\n2. SEED VARIATION (20 samples/class):")
    print("-" * 70)
    print(f"{'Seed':<15} {'Accuracy':<12} {'Class 3':<10} {'Class 6':<10} {'Class 8':<10}")
    print("-" * 70)
    for seed in seeds:
        res = results['seed_experiments'][seed]
        print(f"{seed:<15} {res['accuracy']*100:>10.2f}% {res['per_class'].get(3, 0)*100:>8.2f}% "
              f"{res['per_class'].get(6, 0)*100:>8.2f}% {res['per_class'].get(8, 0)*100:>8.2f}%")
    
    # Statistics
    seed_accuracies = [results['seed_experiments'][s]['accuracy'] for s in seeds]
    mean_acc = np.mean(seed_accuracies)
    std_acc = np.std(seed_accuracies)
    
    print("\n3. SEED VARIATION STATISTICS:")
    print("-" * 70)
    print(f"Mean accuracy: {mean_acc*100:.2f}%")
    print(f"Std deviation: {std_acc*100:.2f}%")
    print(f"Min accuracy: {min(seed_accuracies)*100:.2f}%")
    print(f"Max accuracy: {max(seed_accuracies)*100:.2f}%")
    
    print(f"\n Results saved to: {results_file}")
    print("="*70)
    
    return results

if __name__ == "__main__":
    results = main()

Using device: cuda
GP COMPREHENSIVE EXPERIMENTS

Loading data...

EXPERIMENT 1: Varying Data Size

--- Running with 100 samples per class (seed=42) ---
Accuracy: 93.93%
Per-class (hard): Class 3: 88.80%, Class 6: 89.60%, Class 8: 90.40%

--- Running with 50 samples per class (seed=42) ---
Accuracy: 92.30%
Per-class (hard): Class 3: 86.80%, Class 6: 86.40%, Class 8: 82.80%

--- Running with 20 samples per class (seed=42) ---
Accuracy: 86.96%
Per-class (hard): Class 3: 73.20%, Class 6: 70.80%, Class 8: 64.00%

--- Running with 10 samples per class (seed=42) ---
Accuracy: 82.59%
Per-class (hard): Class 3: 66.40%, Class 6: 57.20%, Class 8: 58.80%

--- Running with 5 samples per class (seed=42) ---
Accuracy: 74.22%
Per-class (hard): Class 3: 58.80%, Class 6: 35.20%, Class 8: 36.40%

--- Running with 1 samples per class (seed=42) ---
Accuracy: 40.22%
Per-class (hard): Class 3: 31.60%, Class 6: 13.60%, Class 8: 17.20%

EXPERIMENT 2: Varying Seed (20 samples/class)

--- Running with seed=42 (2

In [5]:
"""
Smart Attention Initialization: Making Attention "Pre-Tuned"

Core Question: Can we initialize attention mechanisms so intelligently that they 
work well out-of-the-box with minimal/no fine-tuning?

Key Insight: The problem with random initialization is it doesn't encode 
any prior knowledge about what "good" attention should look like.

Multiple Approaches Explored:
1. Task-Specific Priors (spatial locality, semantic similarity)
2. Meta-Learned Initialization (learn to initialize from many tasks)
3. GP-Based Initialization (sample from prior over attention patterns)
4. Symmetric/Structured Initialization (group theory, geometric priors)
5. Curriculum Initialization (start simple, gradually complex)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==================== Approach 1: Spatial Prior Initialization ====================
class SpatialPriorAttention(nn.Module):
    """
    Initialize attention with strong spatial locality bias
    Idea: Nearby tokens should attend to each other more
    Good for: Images, sequences with local structure
    """
    def __init__(self, dim, num_heads=8, spatial_temperature=1.0):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # Standard QKV projections
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Spatial prior parameters (learnable but well-initialized)
        self.spatial_temperature = nn.Parameter(torch.tensor(spatial_temperature))
        self.locality_bias = nn.Parameter(torch.zeros(1))  # How much to bias towards local
        
    def create_spatial_prior(self, seq_len, grid_size=None):
        """
        Create spatial distance-based prior
        For images: 2D spatial distance
        For sequences: 1D distance
        """
        if grid_size is not None:
            # 2D spatial prior (for images arranged as patches)
            h, w = grid_size
            y_coords = torch.arange(h, device=self.spatial_temperature.device).repeat_interleave(w)
            x_coords = torch.arange(w, device=self.spatial_temperature.device).repeat(h)
            
            coords = torch.stack([y_coords, x_coords], dim=1).float()  # (seq_len, 2)
            
            # Compute pairwise distances
            dist = torch.cdist(coords, coords)  # (seq_len, seq_len)
        else:
            # 1D sequential prior
            positions = torch.arange(seq_len, device=self.spatial_temperature.device).float()
            dist = torch.abs(positions.unsqueeze(1) - positions.unsqueeze(0))
        
        # Convert distance to similarity (closer = higher attention)
        # Use RBF-like kernel: exp(-dist^2 / temperature)
        spatial_prior = torch.exp(-dist**2 / (2 * self.spatial_temperature**2))
        
        return spatial_prior
    
    def forward(self, x, grid_size=None):
        """
        x: (batch, seq_len, dim)
        grid_size: (height, width) if input is image patches
        """
        B, N, C = x.shape
        
        # Standard attention computation
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Compute attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, N, N)
        
        # ADD SPATIAL PRIOR
        spatial_prior = self.create_spatial_prior(N, grid_size)  # (N, N)
        spatial_prior = spatial_prior.unsqueeze(0).unsqueeze(0)  # (1, 1, N, N)
        
        # Combine: learned attention + spatial prior
        attn = attn + self.locality_bias * torch.log(spatial_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 2: Meta-Learned Initialization ====================
class MetaLearnedAttention(nn.Module):
    """
    Learn the initialization from many related tasks (meta-learning)
    Idea: Find initialization that's good across many tasks with few updates
    Similar to MAML but for attention weights
    """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # Meta-learned initialization for QKV
        # These are "good starting points" learned from many tasks
        self.meta_qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Meta-learned attention pattern templates
        # These capture common attention patterns across tasks
        self.num_templates = 4
        self.attention_templates = nn.Parameter(torch.randn(self.num_templates, num_heads, 1, 1))
        self.template_weights = nn.Parameter(torch.ones(self.num_templates) / self.num_templates)
        
    def get_meta_attention_prior(self, seq_len):
        """
        Combine learned attention templates
        Each template represents a common pattern (e.g., local, global, skip)
        """
        # Create base patterns
        templates = []
        
        # Template 1: Local attention (diagonal band)
        local = torch.eye(seq_len, device=self.attention_templates.device)
        if seq_len > 1:
            local += torch.diag(torch.ones(seq_len - 1, device=self.attention_templates.device), diagonal=1)
            local += torch.diag(torch.ones(seq_len - 1, device=self.attention_templates.device), diagonal=-1)
        templates.append(local)
        
        # Template 2: Global attention (uniform)
        global_attn = torch.ones(seq_len, seq_len, device=self.attention_templates.device) / seq_len
        templates.append(global_attn)
        
        # Template 3: Skip attention (every other)
        skip = torch.zeros(seq_len, seq_len, device=self.attention_templates.device)
        skip[::2, ::2] = 1.0
        skip = skip / (skip.sum(dim=-1, keepdim=True) + 1e-8)
        templates.append(skip)
        
        # Template 4: Hierarchical (first token attends to all)
        hierarchical = torch.zeros(seq_len, seq_len, device=self.attention_templates.device)
        hierarchical[0, :] = 1.0 / seq_len  # First token = global
        hierarchical[1:, 1:] = torch.eye(seq_len - 1, device=self.attention_templates.device)  # Rest = local
        templates.append(hierarchical)
        
        # Combine templates with learned weights
        templates = torch.stack(templates)  # (num_templates, seq_len, seq_len)
        weights = F.softmax(self.template_weights, dim=0)
        
        combined = (templates * weights.view(-1, 1, 1)).sum(dim=0)
        
        return combined
    
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.meta_qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add meta-learned prior
        meta_prior = self.get_meta_attention_prior(N)  # (N, N)
        meta_prior = meta_prior.unsqueeze(0).unsqueeze(0)  # (1, 1, N, N)
        
        # Soft combination: learned + meta prior
        attn = attn + torch.log(meta_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 3: GP Prior on Attention ====================
class GPPriorAttention(nn.Module):
    """
    Sample attention patterns from Gaussian Process prior
    Idea: GP captures smooth attention patterns with uncertainty
    Good for: When you want structured but stochastic attention
    """
    def __init__(self, dim, num_heads=8, lengthscale=5.0):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # GP hyperparameters
        self.lengthscale = nn.Parameter(torch.tensor(lengthscale))
        self.outputscale = nn.Parameter(torch.tensor(1.0))
        
        # Cached GP samples (can resample during training)
        self.register_buffer('gp_attention_bias', None)
        
    def rbf_kernel(self, positions):
        """RBF kernel for position similarities"""
        dist = torch.cdist(positions, positions)
        K = self.outputscale * torch.exp(-0.5 * dist**2 / self.lengthscale**2)
        return K
    
    def sample_gp_attention_pattern(self, seq_len):
        """
        Sample attention bias from GP
        This gives structured randomness
        """
        positions = torch.arange(seq_len, device=self.lengthscale.device).float().unsqueeze(1)
        
        K = self.rbf_kernel(positions)  # (seq_len, seq_len)
        K = K + 1e-4 * torch.eye(seq_len, device=K.device)
        
        # Cholesky decomposition
        L = torch.linalg.cholesky(K)
        
        # Sample from GP: f = L @ z where z ~ N(0, I)
        z = torch.randn(seq_len, seq_len, device=K.device)
        gp_sample = L @ z
        
        return gp_sample
    
    def forward(self, x, resample_gp=False):
        B, N, C = x.shape
        
        # Sample GP pattern if needed
        if self.gp_attention_bias is None or resample_gp:
            self.gp_attention_bias = self.sample_gp_attention_pattern(N)
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add GP-sampled bias
        gp_bias = self.gp_attention_bias.unsqueeze(0).unsqueeze(0) * 0.1  # Scale down
        attn = attn + gp_bias
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 4: Symmetric/Geometric Initialization ====================
class SymmetricAttention(nn.Module):
    """
    Initialize with geometric/group-theoretic structure
    Idea: Exploit symmetries in the task (rotation, translation invariance)
    Good for: Images, graphs with known symmetries
    """
    def __init__(self, dim, num_heads=8, symmetry_type='rotational'):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.symmetry_type = symmetry_type
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Symmetry-preserving bias
        self.symmetry_strength = nn.Parameter(torch.tensor(1.0))
        
    def create_symmetric_prior(self, seq_len, grid_size=None):
        """
        Create attention pattern with specific symmetry
        """
        if self.symmetry_type == 'rotational' and grid_size is not None:
            # Rotational symmetry for 2D grids
            h, w = grid_size
            center = torch.tensor([h/2, w/2], device=self.symmetry_strength.device)
            
            # Create coordinate grid
            y = torch.arange(h, device=center.device).repeat_interleave(w)
            x = torch.arange(w, device=center.device).repeat(h)
            coords = torch.stack([y, x], dim=1).float()
            
            # Distance from center
            dist_from_center = torch.norm(coords - center, dim=1)
            
            # Attention decays with distance from center (rotationally symmetric)
            similarity = torch.exp(-dist_from_center.unsqueeze(1) * dist_from_center.unsqueeze(0) / 10.0)
            
        elif self.symmetry_type == 'translational':
            # Translation invariant: Toeplitz matrix
            positions = torch.arange(seq_len, device=self.symmetry_strength.device).float()
            relative_pos = positions.unsqueeze(1) - positions.unsqueeze(0)
            
            # Attention depends only on relative position
            similarity = torch.exp(-relative_pos.abs() / 5.0)
            
        else:
            # Default: permutation invariant (all equal)
            similarity = torch.ones(seq_len, seq_len, device=self.symmetry_strength.device) / seq_len
        
        return similarity
    
    def forward(self, x, grid_size=None):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add symmetric prior
        symmetric_prior = self.create_symmetric_prior(N, grid_size)
        symmetric_prior = symmetric_prior.unsqueeze(0).unsqueeze(0)
        
        attn = attn + self.symmetry_strength * torch.log(symmetric_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 5: Curriculum Initialization ====================
class CurriculumAttention(nn.Module):
    """
    Start with simple attention (e.g., uniform), gradually allow complexity
    Idea: Like curriculum learning - start easy, get harder
    Good for: Training from scratch with small data
    """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Curriculum parameter: 0 = simple (uniform), 1 = complex (learned)
        self.complexity = nn.Parameter(torch.tensor(0.0))
        
    def get_curriculum_prior(self, seq_len):
        """
        Interpolate between simple and complex patterns
        """
        # Simple: uniform attention
        uniform = torch.ones(seq_len, seq_len, device=self.complexity.device) / seq_len
        
        # Medium: local attention
        local = torch.eye(seq_len, device=self.complexity.device)
        if seq_len > 1:
            local += 0.5 * torch.diag(torch.ones(seq_len - 1, device=self.complexity.device), diagonal=1)
            local += 0.5 * torch.diag(torch.ones(seq_len - 1, device=self.complexity.device), diagonal=-1)
        local = local / local.sum(dim=-1, keepdim=True)
        
        # Interpolate based on curriculum stage
        complexity = torch.sigmoid(self.complexity)
        
        if complexity < 0.5:
            # Stage 1: uniform -> local
            alpha = complexity * 2
            prior = (1 - alpha) * uniform + alpha * local
        else:
            # Stage 2: local -> learned (no prior)
            alpha = (complexity - 0.5) * 2
            prior = (1 - alpha) * local + alpha * torch.ones_like(local) / seq_len
        
        return prior
    
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add curriculum prior
        curriculum_prior = self.get_curriculum_prior(N)
        curriculum_prior = curriculum_prior.unsqueeze(0).unsqueeze(0)
        
        # Strength of prior decreases as we learn
        prior_strength = 5.0 * (1.0 - torch.sigmoid(self.complexity))
        attn = attn + prior_strength * torch.log(curriculum_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Visualization & Demo ====================
def visualize_attention_patterns():
    """Compare attention patterns from different initialization strategies"""
    seq_len = 16
    dim = 64
    batch = 1
    
    x = torch.randn(batch, seq_len, dim)
    
    models = {
        'Spatial Prior': SpatialPriorAttention(dim),
        'Meta-Learned': MetaLearnedAttention(dim),
        'GP Prior': GPPriorAttention(dim),
        'Symmetric': SymmetricAttention(dim, symmetry_type='translational'),
        'Curriculum': CurriculumAttention(dim),
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (name, model) in enumerate(models.items()):
        if idx < len(axes):
            with torch.no_grad():
                _, attn = model(x)
                
            # Average over heads and batch
            attn_avg = attn[0].mean(dim=0).numpy()  # (seq_len, seq_len)
            
            axes[idx].imshow(attn_avg, cmap='viridis', aspect='auto')
            axes[idx].set_title(f'{name}\nAttention Pattern')
            axes[idx].set_xlabel('Key Position')
            axes[idx].set_ylabel('Query Position')
            axes[idx].colorbar = plt.colorbar(axes[idx].images[0], ax=axes[idx])
    
    # Hide last subplot if odd number
    if len(models) < len(axes):
        axes[-1].axis('off')
    
    plt.tight_layout()
    plt.savefig('/tmp/attention_initialization_comparison.png', dpi=150, bbox_inches='tight')
    print("Visualization saved!")

def demo():
    print("="*70)
    print("Smart Attention Initialization Strategies")
    print("="*70)
    
    print("\n🎯 GOAL: Initialize attention so well it barely needs tuning!\n")
    
    seq_len = 16
    dim = 64
    x = torch.randn(1, seq_len, dim)
    
    print("="*70)
    print("1. SPATIAL PRIOR ATTENTION")
    print("="*70)
    print("Idea: Nearby tokens should attend to each other")
    print("Prior: exp(-distance^2 / temperature)")
    spatial = SpatialPriorAttention(dim)
    out1, attn1 = spatial(x, grid_size=(4, 4))
    print(f"✓ Output shape: {out1.shape}")
    print(f"✓ Spatial temperature: {spatial.spatial_temperature.item():.2f}")
    print(f"✓ Locality bias: {spatial.locality_bias.item():.2f}")
    
    print("\n" + "="*70)
    print("2. META-LEARNED INITIALIZATION")
    print("="*70)
    print("Idea: Learn initialization from many tasks (MAML-style)")
    print("Templates: Local, Global, Skip, Hierarchical")
    meta = MetaLearnedAttention(dim)
    out2, attn2 = meta(x)
    weights = F.softmax(meta.template_weights, dim=0)
    print(f"✓ Output shape: {out2.shape}")
    print(f"✓ Template weights: {weights.detach().numpy()}")
    
    print("\n" + "="*70)
    print("3. GP PRIOR ATTENTION")
    print("="*70)
    print("Idea: Sample structured attention from Gaussian Process")
    print("Kernel: RBF with learned lengthscale")
    gp = GPPriorAttention(dim)
    out3, attn3 = gp(x, resample_gp=True)
    print(f"✓ Output shape: {out3.shape}")
    print(f"✓ GP lengthscale: {gp.lengthscale.item():.2f}")
    print(f"✓ GP outputscale: {gp.outputscale.item():.2f}")
    
    print("\n" + "="*70)
    print("4. SYMMETRIC/GEOMETRIC ATTENTION")
    print("="*70)
    print("Idea: Exploit task symmetries (rotation, translation)")
    print("Prior: Distance from center (rotational symmetry)")
    symmetric = SymmetricAttention(dim, symmetry_type='translational')
    out4, attn4 = symmetric(x)
    print(f"✓ Output shape: {out4.shape}")
    print(f"✓ Symmetry type: {symmetric.symmetry_type}")
    print(f"✓ Symmetry strength: {symmetric.symmetry_strength.item():.2f}")
    
    print("\n" + "="*70)
    print("5. CURRICULUM ATTENTION")
    print("="*70)
    print("Idea: Start simple (uniform), gradually allow complexity")
    print("Stages: Uniform → Local → Learned")
    curriculum = CurriculumAttention(dim)
    out5, attn5 = curriculum(x)
    print(f"✓ Output shape: {out5.shape}")
    print(f"✓ Complexity stage: {torch.sigmoid(curriculum.complexity).item():.2f}")
    
    print("\n" + "="*70)
    print("KEY INSIGHTS")
    print("="*70)
    print("✓ Random init has NO inductive bias → needs lots of data")
    print("✓ Smart init encodes task structure → works with few samples")
    print("✓ Can combine approaches (e.g., Spatial + Meta-learned)")
    print("✓ For YOUR EuroSAT: Spatial prior perfect for satellite images!")
    print()
    print("NEXT STEPS:")
    print("→ Replace ViT attention with SpatialPriorAttention")
    print("→ Should work better with 20 samples (encodes locality bias)")
    print("→ Can still fine-tune, but starts from better place")
    print("="*70)

if __name__ == "__main__":
    demo()
    # visualize_attention_patterns()

Smart Attention Initialization Strategies

🎯 GOAL: Initialize attention so well it barely needs tuning!

1. SPATIAL PRIOR ATTENTION
Idea: Nearby tokens should attend to each other
Prior: exp(-distance^2 / temperature)
✓ Output shape: torch.Size([1, 16, 64])
✓ Spatial temperature: 1.00
✓ Locality bias: 0.00

2. META-LEARNED INITIALIZATION
Idea: Learn initialization from many tasks (MAML-style)
Templates: Local, Global, Skip, Hierarchical
✓ Output shape: torch.Size([1, 16, 64])
✓ Template weights: [0.25 0.25 0.25 0.25]

3. GP PRIOR ATTENTION
Idea: Sample structured attention from Gaussian Process
Kernel: RBF with learned lengthscale
✓ Output shape: torch.Size([1, 16, 64])
✓ GP lengthscale: 5.00
✓ GP outputscale: 1.00

4. SYMMETRIC/GEOMETRIC ATTENTION
Idea: Exploit task symmetries (rotation, translation)
Prior: Distance from center (rotational symmetry)
✓ Output shape: torch.Size([1, 16, 64])
✓ Symmetry type: translational
✓ Symmetry strength: 1.00

5. CURRICULUM ATTENTION
Idea: Start sim

In [6]:
# ==================== Load Data and Setup (from previous cell) ====================
print("="*70)
print("DEEP KERNEL LEARNING EXPERIMENT")
print("="*70)

# Load data
print("\nLoading data...")
train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')
test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')

# Setup labels
if 'ClassName' in train_df.columns:
    classes = sorted(train_df['ClassName'].unique())
    train_df['label_idx'] = train_df['ClassName'].map({c: i for i, c in enumerate(classes)})
    test_df['label_idx'] = test_df['ClassName'].map({c: i for i, c in enumerate(classes)}) if 'ClassName' in test_df.columns else test_df['Label']
else:
    train_df['label_idx'] = train_df['Label']
    test_df['label_idx'] = test_df['Label']

num_classes = len(train_df['label_idx'].unique())
print(f"Number of classes: {num_classes}")

# Subset (20 per class)
train_subset = pd.concat([
    train_df[train_df['label_idx'] == i].sample(n=20, random_state=42)
    for i in range(num_classes)
], ignore_index=True)

print(f"Training samples: {len(train_subset)} ({len(train_subset)//num_classes} per class)")
print(f"Test samples: {len(test_df)}")

# Extract arrays
train_files = train_subset['Filename'].values
train_labels = train_subset['label_idx'].values
test_files = test_df['Filename'].values
test_labels = test_df['label_idx'].values

image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'

# Load ViT
print("\nLoading ViT...")
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

import transformers
transformers.logging.set_verbosity_error()
vit = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)
transformers.logging.set_verbosity_warning()

for p in vit.parameters():
    p.requires_grad = False

# Datasets
train_ds = EuroSATDataset(train_files, train_labels, image_dir, processor)
test_ds = EuroSATDataset(test_files, test_labels, image_dir, processor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Extract features
print("\n" + "="*70)
print("Feature Extraction")
print("="*70)
train_feat, train_lbl = extract_features(train_loader, vit, device)
test_feat, test_lbl = extract_features(test_loader, vit, device)

print(f"Extracted: Train {train_feat.shape}, Test {test_feat.shape}")

# Preprocess
print("\nPreprocessing...")
scaler = StandardScaler()
train_feat = scaler.fit_transform(train_feat)
test_feat = scaler.transform(test_feat)

# PCA
pca_components = 128
pca = PCA(n_components=pca_components, random_state=42)
train_feat = pca.fit_transform(train_feat)
test_feat = pca.transform(test_feat)

print(f"PCA: {pca_components} components, {pca.explained_variance_ratio_.sum():.4f} variance")

# Convert to tensors
train_x = torch.tensor(train_feat, dtype=torch.float32).to(device)
train_y = torch.tensor(train_lbl, dtype=torch.long).to(device)
test_x = torch.tensor(test_feat, dtype=torch.float32).to(device)

# ==================== Adaptive Deep Kernel Learning ====================
class AdaptiveDKL(nn.Module):
    def __init__(self, input_dim=128, hidden_dims=[256, 128, 64]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev_dim = hidden_dim
        
        self.feature_extractor = nn.Sequential(*layers)
        self.output_dim = hidden_dims[-1]
        
    def forward(self, x):
        return self.feature_extractor(x)

class DKLGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, train_x, train_y, feature_extractor, num_inducing=100):
        pos_indices = torch.where(train_y == 1)[0]
        neg_indices = torch.where(train_y == 0)[0]
        
        num_pos = min(num_inducing // 2, len(pos_indices))
        num_neg = num_inducing - num_pos
        
        if len(pos_indices) > 0 and len(neg_indices) > 0:
            pos_sample = pos_indices[torch.randperm(len(pos_indices))[:num_pos]]
            neg_sample = neg_indices[torch.randperm(len(neg_indices))[:num_neg]]
            inducing_idx = torch.cat([pos_sample, neg_sample])
        else:
            inducing_idx = torch.randperm(train_x.size(0))[:num_inducing]
        
        inducing_points = train_x[inducing_idx].detach().clone()
        
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=inducing_points.size(0)
        )
        
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution,
            learn_inducing_locations=True
        )
        
        super().__init__(variational_strategy)
        
        self.feature_extractor = feature_extractor
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel() + 
            gpytorch.kernels.LinearKernel()
        )
        
    def forward(self, x):
        projected_x = self.feature_extractor(x)
        mean_x = self.mean_module(projected_x)
        covar_x = self.covar_module(projected_x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# ==================== DKL One-vs-Rest Classifier ====================
class DKLOneVsRestGP:
    def __init__(self, num_classes, input_dim=128, num_inducing=130):
        self.num_classes = num_classes
        self.input_dim = input_dim
        self.num_inducing = num_inducing
        self.models = []
        self.likelihoods = []
        self.feature_extractors = []
        self.class_priors = []
    
    def fit(self, train_x, train_y, n_epochs=150, initial_lr=0.001):
        """Train DKL GPs"""
        for class_idx in range(self.num_classes):
            print(f"\nTraining DKL-GP for class {class_idx}...")
            
            # Binary labels
            train_y_binary = (train_y == class_idx).float()
            
            num_pos = train_y_binary.sum().item()
            num_neg = (train_y_binary == 0).sum().item()
            print(f"  Positive: {num_pos}, Negative: {num_neg}")
            
            # Store class prior
            self.class_priors.append(num_pos / len(train_y))
            
            # Create DKL feature extractor
            feature_extractor = AdaptiveDKL(
                input_dim=self.input_dim, 
                hidden_dims=[256, 128, 64]
            ).to(train_x.device)
            
            # Create DKL GP
            model = DKLGPModel(
                train_x, train_y_binary, feature_extractor,
                num_inducing=self.num_inducing
            ).to(train_x.device)
            
            likelihood = gpytorch.likelihoods.BernoulliLikelihood().to(train_x.device)
            
            # Train
            model.train()
            likelihood.train()
            
            # FIXED: Properly separate parameters using sets to avoid overlap
            feature_extractor_params = set(model.feature_extractor.parameters())
            variational_params = set(model.variational_parameters())
            likelihood_params = set(likelihood.parameters())
            
            # GP hyperparameters = all model params - feature extractor - variational
            all_model_params = set(model.parameters())
            gp_hyper_params = all_model_params - feature_extractor_params - variational_params
            
            optimizer = torch.optim.Adam([
                {'params': list(feature_extractor_params), 'lr': initial_lr, 'weight_decay': 1e-4},
                {'params': list(variational_params), 'lr': initial_lr * 10},
                {'params': list(gp_hyper_params), 'lr': initial_lr * 5},
                {'params': list(likelihood_params), 'lr': initial_lr * 5},
            ])
            
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=15, verbose=False
            )
            
            mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
            
            best_loss = float('inf')
            patience = 0
            max_patience = 30
            
            for epoch in range(n_epochs):
                optimizer.zero_grad()
                output = model(train_x)
                loss = -mll(output, train_y_binary)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                loss_val = loss.item()
                scheduler.step(loss_val)
                
                if loss_val < best_loss:
                    best_loss = loss_val
                    patience = 0
                else:
                    patience += 1
                
                if patience >= max_patience:
                    break
            
            print(f"  Final loss: {best_loss:.4f}")
            
            self.models.append(model)
            self.likelihoods.append(likelihood)
            self.feature_extractors.append(feature_extractor)
    
    def predict(self, test_x, calibrate=True, batch_size=100):
        """Predict with DKL"""
        print("\nMaking DKL predictions...")
        
        for model, likelihood in zip(self.models, self.likelihoods):
            model.eval()
            likelihood.eval()
        
        all_scores = []
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            for class_idx in tqdm(range(self.num_classes), desc="Computing DKL scores"):
                model = self.models[class_idx]
                likelihood = self.likelihoods[class_idx]
                
                class_scores = []
                
                for i in range(0, test_x.size(0), batch_size):
                    batch_x = test_x[i:min(i+batch_size, test_x.size(0))]
                    
                    output = model(batch_x)
                    pred = likelihood(output)
                    scores = pred.mean
                    class_scores.append(scores.cpu())
                
                class_scores = torch.cat(class_scores)
                all_scores.append(class_scores)
        
        all_scores = torch.stack(all_scores)
        
        if calibrate:
            priors = torch.tensor(self.class_priors).unsqueeze(1)
            all_scores = all_scores * priors
        
        predictions = all_scores.argmax(dim=0).numpy()
        
        return predictions, all_scores.numpy()

# ==================== Train and Evaluate DKL ====================
print("\n" + "="*70)
print("Training DKL-GP Classifier")
print("="*70)

# Train DKL GP
dkl_classifier = DKLOneVsRestGP(
    num_classes=num_classes,
    input_dim=pca_components,  # 128
    num_inducing=130
)

dkl_classifier.fit(
    train_x, train_y,
    n_epochs=150,
    initial_lr=0.001
)

# Predict with DKL
predictions_dkl, scores_dkl = dkl_classifier.predict(
    test_x,
    calibrate=True,
    batch_size=100
)

# Calculate accuracy
accuracy_dkl = accuracy_score(test_lbl, predictions_dkl)

print(f"\n{'='*70}")
print(f"DKL-GP TEST ACCURACY: {accuracy_dkl:.4f} ({accuracy_dkl*100:.2f}%)")
print(f"{'='*70}")

# Detailed report
print("\nDKL Classification Report:")
print("="*70)
print(classification_report(test_lbl, predictions_dkl,
                            target_names=[f"Class {i}" for i in range(num_classes)],
                            digits=4))

# Per-class performance
print("\nDKL Per-Class Performance:")
print("-"*70)
for i in range(num_classes):
    mask = test_lbl == i
    if mask.sum() > 0:
        acc = accuracy_score(test_lbl[mask], predictions_dkl[mask])
        avg_score = scores_dkl[i, mask].mean()
        std_score = scores_dkl[i, mask].std()
        print(f"Class {i:2d}: Acc={acc*100:6.2f}% | "
              f"Score={avg_score:.3f}±{std_score:.3f} | "
              f"N={mask.sum():4d}")

print("\n" + "="*70)
print("DKL FEATURES:")
print("✓ Adaptive deep feature learning (128→256→128→64)")
print("✓ RBF + Linear kernel on learned features")
print("✓ End-to-end training of feature extractor + GP")
print("✓ Batch normalization and dropout for regularization")
print("✓ Class-balanced inducing points")
print("✓ Prior calibration in prediction")
print("="*70)

DEEP KERNEL LEARNING EXPERIMENT

Loading data...
Number of classes: 10
Training samples: 200 (20 per class)
Test samples: 2700

Loading ViT...

Feature Extraction


Extracting features: 100%|██████████| 85/85 [00:40<00:00,  2.11it/s]


Extracted: Train (200, 768), Test (2700, 768)

Preprocessing...
PCA: 128 components, 0.9549 variance

Training DKL-GP Classifier

Training DKL-GP for class 0...
  Positive: 20.0, Negative: 180
  Final loss: 0.1140

Training DKL-GP for class 1...
  Positive: 20.0, Negative: 180
  Final loss: 0.1305

Training DKL-GP for class 2...
  Positive: 20.0, Negative: 180
  Final loss: 0.1463

Training DKL-GP for class 3...
  Positive: 20.0, Negative: 180
  Final loss: 0.1333

Training DKL-GP for class 4...
  Positive: 20.0, Negative: 180
  Final loss: 0.1284

Training DKL-GP for class 5...
  Positive: 20.0, Negative: 180
  Final loss: 0.1567

Training DKL-GP for class 6...
  Positive: 20.0, Negative: 180
  Final loss: 0.1465

Training DKL-GP for class 7...
  Positive: 20.0, Negative: 180
  Final loss: 0.1420

Training DKL-GP for class 8...
  Positive: 20.0, Negative: 180
  Final loss: 0.1298

Training DKL-GP for class 9...
  Positive: 20.0, Negative: 180
  Final loss: 0.1303

Making DKL predictio

Computing DKL scores: 100%|██████████| 10/10 [00:01<00:00,  6.39it/s]


DKL-GP TEST ACCURACY: 0.8370 (83.70%)

DKL Classification Report:
              precision    recall  f1-score   support

     Class 0     0.8338    0.9533    0.8896       300
     Class 1     0.8584    0.9500    0.9019       300
     Class 2     0.8449    0.8533    0.8491       300
     Class 3     0.7685    0.6240    0.6887       250
     Class 4     0.8453    0.9400    0.8902       250
     Class 5     0.7039    0.8200    0.7575       200
     Class 6     0.8820    0.6280    0.7336       250
     Class 7     0.8711    0.9233    0.8964       300
     Class 8     0.7767    0.6680    0.7183       250
     Class 9     0.9327    0.9233    0.9280       300

    accuracy                         0.8370      2700
   macro avg     0.8317    0.8283    0.8253      2700
weighted avg     0.8375    0.8370    0.8329      2700


DKL Per-Class Performance:
----------------------------------------------------------------------
Class  0: Acc= 95.33% | Score=0.063±0.011 | N= 300
Class  1: Acc= 95.00% | 

In [7]:
"""
FIXED: Fused ViT + Handcrafted Features GP Classification
==========================================================
Key fixes:
1. Removed problematic multi-kernel active_dims (was causing training collapse)
2. Better feature normalization after fusion
3. More robust GP with simpler kernel
4. Higher learning rates and better optimization
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
from scipy import ndimage, fftpack
from scipy.stats import skew, kurtosis, entropy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report
import gpytorch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check for optional dependencies
try:
    from skimage.feature import local_binary_pattern, hog
    from skimage.filters import gabor_kernel
    from skimage import color
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print("Warning: skimage not available.")

try:
    import pywt
    HAS_PYWT = True
except ImportError:
    HAS_PYWT = False
    print("Warning: pywt not available.")

# ==================== Handcrafted Feature Extractor ====================

class HandcraftedFeatureExtractor:
    """Extract handcrafted features from images."""
    
    def __init__(self, feature_types='all'):
        self.all_features = ['fourier', 'gabor', 'edge', 'lbp', 
                             'hog', 'color_hist', 'wavelet', 'stats']
        
        if feature_types == 'all':
            self.feature_types = self.all_features.copy()
            if not HAS_SKIMAGE:
                self.feature_types = [f for f in self.feature_types 
                                      if f not in ['gabor', 'lbp', 'hog']]
            if not HAS_PYWT:
                self.feature_types = [f for f in self.feature_types if f != 'wavelet']
        else:
            self.feature_types = feature_types
        
        if 'gabor' in self.feature_types and HAS_SKIMAGE:
            self.gabor_kernels = self._create_gabor_bank()
        else:
            self.gabor_kernels = []
    
    def _create_gabor_bank(self):
        kernels = []
        for theta in np.arange(0, np.pi, np.pi / 4):
            for frequency in [0.1, 0.2, 0.4]:
                kernel = np.real(gabor_kernel(frequency, theta=theta, sigma_x=3, sigma_y=3))
                kernels.append(kernel)
        return kernels
    
    def extract_fourier_features(self, img_gray, n_features=24):
        f_transform = fftpack.fft2(img_gray)
        f_shift = fftpack.fftshift(f_transform)
        magnitude = np.abs(f_shift)
        log_magnitude = np.log1p(magnitude)
        
        h, w = img_gray.shape
        center_h, center_w = h // 2, w // 2
        max_radius = min(center_h, center_w)
        
        features = []
        
        n_bands = 8
        for i in range(n_bands):
            r_inner = int(i * max_radius / n_bands)
            r_outer = int((i + 1) * max_radius / n_bands)
            y, x = np.ogrid[:h, :w]
            dist = np.sqrt((x - center_w)**2 + (y - center_h)**2)
            mask = (dist >= r_inner) & (dist < r_outer)
            band_energy = np.mean(log_magnitude[mask]) if mask.sum() > 0 else 0
            features.append(band_energy)
        
        n_sectors = 8
        for i in range(n_sectors):
            angle_start = i * 2 * np.pi / n_sectors - np.pi
            angle_end = (i + 1) * 2 * np.pi / n_sectors - np.pi
            y, x = np.ogrid[:h, :w]
            angles = np.arctan2(y - center_h, x - center_w)
            mask = (angles >= angle_start) & (angles < angle_end)
            sector_energy = np.mean(log_magnitude[mask]) if mask.sum() > 0 else 0
            features.append(sector_energy)
        
        central_mask = np.sqrt((np.arange(w) - center_w)**2 + 
                               (np.arange(h)[:, None] - center_h)**2) < max_radius * 0.3
        central_energy = np.mean(log_magnitude[central_mask])
        peripheral_energy = np.mean(log_magnitude[~central_mask])
        features.append(central_energy / (peripheral_energy + 1e-6))
        
        features.extend([np.mean(log_magnitude), np.std(log_magnitude), np.max(log_magnitude)])
        
        high_freq_mask = np.sqrt((np.arange(w) - center_w)**2 + 
                                  (np.arange(h)[:, None] - center_h)**2) > max_radius * 0.5
        high_freq_ratio = np.sum(magnitude[high_freq_mask]) / (np.sum(magnitude) + 1e-6)
        features.append(high_freq_ratio)
        
        return np.array(features[:n_features])
    
    def extract_gabor_features(self, img_gray):
        features = []
        for kernel in self.gabor_kernels:
            filtered = ndimage.convolve(img_gray, kernel, mode='wrap')
            features.extend([np.mean(filtered), np.std(filtered), 
                           np.max(np.abs(filtered)), np.sum(filtered**2) / filtered.size])
        return np.array(features)
    
    def extract_edge_features(self, img_gray):
        features = []
        
        sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
        sobel_dir = np.arctan2(sobel_y, sobel_x)
        
        features.extend([np.mean(sobel_mag), np.std(sobel_mag), np.max(sobel_mag)])
        
        dir_hist, _ = np.histogram(sobel_dir.flatten(), bins=8, range=(-np.pi, np.pi))
        dir_hist = dir_hist / (dir_hist.sum() + 1e-6)
        features.extend(dir_hist)
        
        laplacian = cv2.Laplacian(img_gray, cv2.CV_64F)
        features.extend([np.mean(np.abs(laplacian)), np.std(laplacian), 
                        np.sum(laplacian**2) / laplacian.size])
        
        img_uint8 = (img_gray * 255).astype(np.uint8)
        edges_canny = cv2.Canny(img_uint8, 50, 150)
        features.append(np.sum(edges_canny > 0) / edges_canny.size)
        
        valid_edges = sobel_mag > np.percentile(sobel_mag, 75)
        if valid_edges.sum() > 10:
            edge_dirs = sobel_dir[valid_edges]
            dir_hist2, _ = np.histogram(edge_dirs, bins=16, range=(-np.pi, np.pi))
            dir_hist2 = dir_hist2 / (dir_hist2.sum() + 1e-6)
            features.append(entropy(dir_hist2))
        else:
            features.append(0)
        
        return np.array(features)
    
    def extract_lbp_features(self, img_gray, radius=2, n_points=16):
        lbp = local_binary_pattern(img_gray, n_points, radius, method='uniform')
        n_bins = n_points + 2
        lbp_hist, _ = np.histogram(lbp.flatten(), bins=n_bins, range=(0, n_bins))
        lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-6)
        
        features = list(lbp_hist)
        features.extend([np.mean(lbp), np.std(lbp), entropy(lbp_hist)])
        return np.array(features)
    
    def extract_hog_features(self, img_gray):
        img_resized = cv2.resize(img_gray, (64, 64))
        hog_features = hog(img_resized, orientations=9, pixels_per_cell=(16, 16),
                          cells_per_block=(2, 2), visualize=False, feature_vector=True)
        return hog_features
    
    def extract_color_histogram_features(self, img_rgb, n_bins=16):
        features = []
        
        for i in range(3):
            hist, _ = np.histogram(img_rgb[:, :, i].flatten(), bins=n_bins, range=(0, 1))
            hist = hist / (hist.sum() + 1e-6)
            features.extend(hist)
        
        if HAS_SKIMAGE:
            img_hsv = color.rgb2hsv(img_rgb)
        else:
            img_hsv = cv2.cvtColor((img_rgb * 255).astype(np.uint8), cv2.COLOR_RGB2HSV) / 255.0
        
        for i in range(3):
            hist, _ = np.histogram(img_hsv[:, :, i].flatten(), bins=n_bins, range=(0, 1))
            hist = hist / (hist.sum() + 1e-6)
            features.extend(hist)
        
        for i in range(3):
            channel = img_rgb[:, :, i].flatten()
            features.extend([np.mean(channel), np.std(channel), skew(channel)])
        
        r, g, b = img_rgb[:, :, 0], img_rgb[:, :, 1], img_rgb[:, :, 2]
        total = r + g + b + 1e-6
        features.extend([np.mean(r / total), np.mean(g / total), np.mean(b / total)])
        
        features.extend([np.mean(img_hsv[:, :, 1]), np.std(img_hsv[:, :, 1]),
                        np.mean(img_hsv[:, :, 2]), np.std(img_hsv[:, :, 2])])
        
        return np.array(features)
    
    def extract_wavelet_features(self, img_gray, wavelet='db4', level=3):
        features = []
        coeffs = pywt.wavedec2(img_gray, wavelet, level=level)
        
        cA = coeffs[0]
        features.extend([np.mean(cA), np.std(cA), np.sum(cA**2) / cA.size])
        
        for i in range(1, len(coeffs)):
            cH, cV, cD = coeffs[i]
            for detail in [cH, cV, cD]:
                features.extend([np.mean(np.abs(detail)), np.std(detail), 
                               np.sum(detail**2) / detail.size])
        
        total_energy = sum(np.sum(c**2) for c in [coeffs[0]] + 
                          [d for level in coeffs[1:] for d in level])
        for i in range(1, len(coeffs)):
            level_energy = sum(np.sum(d**2) for d in coeffs[i])
            features.append(level_energy / (total_energy + 1e-6))
        
        return np.array(features)
    
    def extract_statistical_features(self, img_gray, img_rgb):
        features = []
        
        features.extend([np.mean(img_gray), np.std(img_gray), skew(img_gray.flatten()),
                        kurtosis(img_gray.flatten()), np.median(img_gray),
                        np.percentile(img_gray, 25), np.percentile(img_gray, 75)])
        
        hist, _ = np.histogram(img_gray.flatten(), bins=256, range=(0, 1))
        hist = hist / (hist.sum() + 1e-6)
        features.append(entropy(hist))
        
        features.append(np.sqrt(np.mean((img_gray - np.mean(img_gray))**2)))
        min_val, max_val = np.min(img_gray), np.max(img_gray)
        features.append((max_val - min_val) / (max_val + min_val + 1e-6))
        
        features.append(np.sum(hist**2))
        
        autocorr = np.corrcoef(img_gray[:-1, :].flatten(), img_gray[1:, :].flatten())[0, 1]
        features.append(autocorr if not np.isnan(autocorr) else 0)
        
        r, g, b = img_rgb[:, :, 0], img_rgb[:, :, 1], img_rgb[:, :, 2]
        features.append(np.std([np.mean(r), np.mean(g), np.mean(b)]))
        
        return np.array(features)
    
    def extract_all_features(self, img_rgb):
        if img_rgb.max() > 1:
            img_rgb = img_rgb / 255.0
        
        if HAS_SKIMAGE:
            img_gray = color.rgb2gray(img_rgb)
        else:
            img_gray = cv2.cvtColor((img_rgb * 255).astype(np.uint8), 
                                    cv2.COLOR_RGB2GRAY) / 255.0
        
        all_features = []
        
        if 'fourier' in self.feature_types:
            all_features.append(self.extract_fourier_features(img_gray))
        if 'gabor' in self.feature_types and HAS_SKIMAGE:
            all_features.append(self.extract_gabor_features(img_gray))
        if 'edge' in self.feature_types:
            all_features.append(self.extract_edge_features(img_gray))
        if 'lbp' in self.feature_types and HAS_SKIMAGE:
            all_features.append(self.extract_lbp_features(img_gray))
        if 'hog' in self.feature_types and HAS_SKIMAGE:
            all_features.append(self.extract_hog_features(img_gray))
        if 'color_hist' in self.feature_types:
            all_features.append(self.extract_color_histogram_features(img_rgb))
        if 'wavelet' in self.feature_types and HAS_PYWT:
            all_features.append(self.extract_wavelet_features(img_gray))
        if 'stats' in self.feature_types:
            all_features.append(self.extract_statistical_features(img_gray, img_rgb))
        
        return np.concatenate(all_features)
    
    def extract_from_dataset(self, image_files, image_dir):
        all_features = []
        for filename in tqdm(image_files, desc="Extracting handcrafted features"):
            img_path = f"{image_dir}/{filename}"
            img = cv2.imread(img_path)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
            img_rgb = cv2.resize(img_rgb, (224, 224))
            features = self.extract_all_features(img_rgb)
            all_features.append(features)
        return np.array(all_features)


# ==================== FIXED GP Model (Simple & Robust) ====================

class RobustGP(gpytorch.models.ApproximateGP):
    """Simple GP that actually works - no fancy multi-kernel stuff."""
    
    def __init__(self, train_x, train_y, num_inducing=100):
        # Class-balanced inducing points
        pos_idx = torch.where(train_y == 1)[0]
        neg_idx = torch.where(train_y == 0)[0]
        
        num_pos = min(num_inducing // 2, len(pos_idx))
        num_neg = num_inducing - num_pos
        
        if len(pos_idx) > 0 and len(neg_idx) > 0:
            pos_sample = pos_idx[torch.randperm(len(pos_idx))[:num_pos]]
            neg_sample = neg_idx[torch.randperm(len(neg_idx))[:num_neg]]
            inducing_idx = torch.cat([pos_sample, neg_sample])
        else:
            inducing_idx = torch.randperm(train_x.size(0))[:num_inducing]
        
        inducing_points = train_x[inducing_idx].detach().clone()
        
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=inducing_points.size(0)
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        
        super().__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        
        # Simple but effective kernel: RBF + Matern
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel()
        ) + gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(nu=2.5)
        )
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class RobustGPClassifier:
    """Fixed One-vs-Rest GP Classifier."""
    
    def __init__(self, num_classes, num_inducing=130):
        self.num_classes = num_classes
        self.num_inducing = num_inducing
        self.models = []
        self.likelihoods = []
        self.class_priors = []
    
    def fit(self, train_x, train_y, n_epochs=200, lr=0.05):
        """Train with proper learning rates."""
        
        for class_idx in range(self.num_classes):
            print(f"\nTraining GP for class {class_idx}...")
            
            train_y_binary = (train_y == class_idx).float()
            num_pos = train_y_binary.sum().item()
            print(f"  Positive: {int(num_pos)}, Negative: {int(len(train_y) - num_pos)}")
            
            self.class_priors.append(num_pos / len(train_y))
            
            model = RobustGP(train_x, train_y_binary, self.num_inducing).to(train_x.device)
            likelihood = gpytorch.likelihoods.BernoulliLikelihood().to(train_x.device)
            
            model.train()
            likelihood.train()
            
            # Proper parameter separation using sets
            variational_params = set(model.variational_parameters())
            all_model_params = set(model.parameters())
            hyper_params = all_model_params - variational_params
            
            optimizer = torch.optim.Adam([
                {'params': list(variational_params), 'lr': lr},
                {'params': list(hyper_params), 'lr': lr * 0.1},
                {'params': list(likelihood.parameters()), 'lr': lr * 0.1},
            ])
            
            mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
            
            best_loss = float('inf')
            patience_counter = 0
            best_state = None
            
            for epoch in range(n_epochs):
                optimizer.zero_grad()
                output = model(train_x)
                loss = -mll(output, train_y_binary)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                current_loss = loss.item()
                
                if current_loss < best_loss - 0.001:  # Significant improvement
                    best_loss = current_loss
                    patience_counter = 0
                    best_state = {
                        'model': {k: v.clone() for k, v in model.state_dict().items()},
                        'likelihood': {k: v.clone() for k, v in likelihood.state_dict().items()}
                    }
                else:
                    patience_counter += 1
                
                if patience_counter >= 40:
                    break
            
            # Restore best state
            if best_state is not None:
                model.load_state_dict(best_state['model'])
                likelihood.load_state_dict(best_state['likelihood'])
            
            print(f"  Final loss: {best_loss:.4f}, Epochs: {epoch + 1}")
            
            self.models.append(model)
            self.likelihoods.append(likelihood)
    
    def predict(self, test_x, calibrate=True, batch_size=100):
        for model, likelihood in zip(self.models, self.likelihoods):
            model.eval()
            likelihood.eval()
        
        all_scores = []
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            for class_idx in tqdm(range(self.num_classes), desc="Predicting"):
                model = self.models[class_idx]
                likelihood = self.likelihoods[class_idx]
                
                class_scores = []
                for i in range(0, test_x.size(0), batch_size):
                    batch_x = test_x[i:min(i+batch_size, test_x.size(0))]
                    output = model(batch_x)
                    pred = likelihood(output)
                    class_scores.append(pred.mean.cpu())
                
                all_scores.append(torch.cat(class_scores))
        
        all_scores = torch.stack(all_scores)
        
        if calibrate:
            priors = torch.tensor(self.class_priors).unsqueeze(1)
            all_scores = all_scores * priors
        
        return all_scores.argmax(dim=0).numpy(), all_scores.numpy()


# ==================== MAIN EXPERIMENT ====================

print("="*70)
print("FUSED ViT + HANDCRAFTED FEATURES GP (FIXED)")
print("="*70)

# Load data
print("\nLoading data...")
train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')
test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')

if 'ClassName' in train_df.columns:
    classes = sorted(train_df['ClassName'].unique())
    train_df['label_idx'] = train_df['ClassName'].map({c: i for i, c in enumerate(classes)})
    test_df['label_idx'] = test_df['ClassName'].map({c: i for i, c in enumerate(classes)}) if 'ClassName' in test_df.columns else test_df['Label']
else:
    train_df['label_idx'] = train_df['Label']
    test_df['label_idx'] = test_df['Label']

num_classes = len(train_df['label_idx'].unique())
print(f"Number of classes: {num_classes}")

# Subset (20 per class)
train_subset = pd.concat([
    train_df[train_df['label_idx'] == i].sample(n=20, random_state=42)
    for i in range(num_classes)
], ignore_index=True)

print(f"Training samples: {len(train_subset)} ({len(train_subset)//num_classes} per class)")
print(f"Test samples: {len(test_df)}")

train_files = train_subset['Filename'].values
train_labels = train_subset['label_idx'].values
test_files = test_df['Filename'].values
test_labels = test_df['label_idx'].values
image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'

# Load ViT
print("\nLoading ViT...")
from transformers import ViTImageProcessor, ViTModel
import transformers
transformers.logging.set_verbosity_error()

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
vit = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)
for p in vit.parameters():
    p.requires_grad = False

transformers.logging.set_verbosity_warning()

# Dataset class
class EuroSATDataset(Dataset):
    def __init__(self, files, labels, img_dir, proc):
        self.files = files
        self.labels = labels
        self.img_dir = img_dir
        self.proc = proc
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        img_path = f"{self.img_dir}/{self.files[idx]}"
        img = Image.open(img_path).convert('RGB')
        inputs = self.proc(images=img, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0), self.labels[idx]

# Extract ViT features
print("\n" + "="*70)
print("1. EXTRACTING ViT FEATURES")
print("="*70)

def extract_vit_features(loader, model, dev):
    features, labels = [], []
    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in tqdm(loader, desc="ViT features"):
            batch_x = batch_x.to(dev)
            outputs = model(batch_x)
            feat = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            features.append(feat)
            labels.append(batch_y.numpy())
    return np.vstack(features), np.concatenate(labels)

train_ds = EuroSATDataset(train_files, train_labels, image_dir, processor)
test_ds = EuroSATDataset(test_files, test_labels, image_dir, processor)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

train_vit_feat, train_lbl = extract_vit_features(train_loader, vit, device)
test_vit_feat, test_lbl = extract_vit_features(test_loader, vit, device)

print(f"ViT features: Train {train_vit_feat.shape}, Test {test_vit_feat.shape}")

# Extract Handcrafted features
print("\n" + "="*70)
print("2. EXTRACTING HANDCRAFTED FEATURES")
print("="*70)

hc_extractor = HandcraftedFeatureExtractor(feature_types='all')
print(f"Feature types: {hc_extractor.feature_types}")

train_hc_feat = hc_extractor.extract_from_dataset(train_files, image_dir)
test_hc_feat = hc_extractor.extract_from_dataset(test_files, image_dir)

# Handle NaN/Inf
train_hc_feat = np.nan_to_num(train_hc_feat, nan=0.0, posinf=1e6, neginf=-1e6)
test_hc_feat = np.nan_to_num(test_hc_feat, nan=0.0, posinf=1e6, neginf=-1e6)

print(f"Handcrafted features: Train {train_hc_feat.shape}, Test {test_hc_feat.shape}")

# ==================== APPROACH 1: Fused Features ====================
print("\n" + "="*70)
print("3. FUSING FEATURES (ViT + Handcrafted)")
print("="*70)

# Standardize ViT
scaler_vit = StandardScaler()
train_vit_scaled = scaler_vit.fit_transform(train_vit_feat)
test_vit_scaled = scaler_vit.transform(test_vit_feat)

# Standardize Handcrafted
scaler_hc = StandardScaler()
train_hc_scaled = scaler_hc.fit_transform(train_hc_feat)
test_hc_scaled = scaler_hc.transform(test_hc_feat)

# Concatenate BEFORE PCA (joint dimensionality reduction)
train_combined = np.concatenate([train_vit_scaled, train_hc_scaled], axis=1)
test_combined = np.concatenate([test_vit_scaled, test_hc_scaled], axis=1)

print(f"Combined features before PCA: {train_combined.shape}")

# Joint PCA
TOTAL_PCA = 128
pca_combined = PCA(n_components=TOTAL_PCA, random_state=42)
train_fused = pca_combined.fit_transform(train_combined)
test_fused = pca_combined.transform(test_combined)

print(f"Joint PCA: {TOTAL_PCA} components, {pca_combined.explained_variance_ratio_.sum():.4f} variance")
print(f"Fused features: Train {train_fused.shape}, Test {test_fused.shape}")

# Final standardization after PCA (important!)
scaler_final = StandardScaler()
train_fused = scaler_final.fit_transform(train_fused)
test_fused = scaler_final.transform(test_fused)

# Convert to tensors
train_x_fused = torch.tensor(train_fused, dtype=torch.float32).to(device)
train_y = torch.tensor(train_lbl, dtype=torch.long).to(device)
test_x_fused = torch.tensor(test_fused, dtype=torch.float32).to(device)

# Train Fused GP
print("\n" + "="*70)
print("4. TRAINING FUSED FEATURE GP")
print("="*70)

classifier_fused = RobustGPClassifier(num_classes=num_classes, num_inducing=150)
classifier_fused.fit(train_x_fused, train_y, n_epochs=200, lr=0.05)

predictions_fused, scores_fused = classifier_fused.predict(test_x_fused, calibrate=True)
accuracy_fused = accuracy_score(test_lbl, predictions_fused)

print(f"\n{'='*70}")
print(f"FUSED FEATURE GP ACCURACY: {accuracy_fused:.4f} ({accuracy_fused*100:.2f}%)")
print(f"{'='*70}")

# ==================== APPROACH 2: ViT Only (Baseline) ====================
print("\n" + "="*70)
print("5. VIT-ONLY GP (BASELINE)")
print("="*70)

# PCA on ViT only
pca_vit = PCA(n_components=128, random_state=42)
train_vit_pca = pca_vit.fit_transform(train_vit_scaled)
test_vit_pca = pca_vit.transform(test_vit_scaled)

scaler_vit_final = StandardScaler()
train_vit_pca = scaler_vit_final.fit_transform(train_vit_pca)
test_vit_pca = scaler_vit_final.transform(test_vit_pca)

train_x_vit = torch.tensor(train_vit_pca, dtype=torch.float32).to(device)
test_x_vit = torch.tensor(test_vit_pca, dtype=torch.float32).to(device)

classifier_vit = RobustGPClassifier(num_classes=num_classes, num_inducing=150)
classifier_vit.fit(train_x_vit, train_y, n_epochs=200, lr=0.05)

predictions_vit, scores_vit = classifier_vit.predict(test_x_vit, calibrate=True)
accuracy_vit = accuracy_score(test_lbl, predictions_vit)

print(f"\n{'='*70}")
print(f"VIT-ONLY GP ACCURACY: {accuracy_vit:.4f} ({accuracy_vit*100:.2f}%)")
print(f"{'='*70}")

# ==================== APPROACH 3: Handcrafted Only ====================
print("\n" + "="*70)
print("6. HANDCRAFTED-ONLY GP")
print("="*70)

# PCA on Handcrafted only
pca_hc = PCA(n_components=min(128, train_hc_scaled.shape[1]), random_state=42)
train_hc_pca = pca_hc.fit_transform(train_hc_scaled)
test_hc_pca = pca_hc.transform(test_hc_scaled)

scaler_hc_final = StandardScaler()
train_hc_pca = scaler_hc_final.fit_transform(train_hc_pca)
test_hc_pca = scaler_hc_final.transform(test_hc_pca)

train_x_hc = torch.tensor(train_hc_pca, dtype=torch.float32).to(device)
test_x_hc = torch.tensor(test_hc_pca, dtype=torch.float32).to(device)

classifier_hc = RobustGPClassifier(num_classes=num_classes, num_inducing=150)
classifier_hc.fit(train_x_hc, train_y, n_epochs=200, lr=0.05)

predictions_hc, scores_hc = classifier_hc.predict(test_x_hc, calibrate=True)
accuracy_hc = accuracy_score(test_lbl, predictions_hc)

print(f"\n{'='*70}")
print(f"HANDCRAFTED-ONLY GP ACCURACY: {accuracy_hc:.4f} ({accuracy_hc*100:.2f}%)")
print(f"{'='*70}")

# ==================== APPROACH 4: Ensemble (Score Averaging) ====================
print("\n" + "="*70)
print("7. ENSEMBLE (VIT + HANDCRAFTED SCORE FUSION)")
print("="*70)

# Normalize scores and combine
scores_vit_norm = scores_vit / (np.abs(scores_vit).max(axis=0, keepdims=True) + 1e-6)
scores_hc_norm = scores_hc / (np.abs(scores_hc).max(axis=0, keepdims=True) + 1e-6)

# Weighted average (ViT typically stronger)
ensemble_scores = 0.7 * scores_vit_norm + 0.3 * scores_hc_norm
predictions_ensemble = ensemble_scores.argmax(axis=0)
accuracy_ensemble = accuracy_score(test_lbl, predictions_ensemble)

print(f"ENSEMBLE GP ACCURACY: {accuracy_ensemble:.4f} ({accuracy_ensemble*100:.2f}%)")

# ==================== FINAL COMPARISON ====================
print("\n" + "="*70)
print("FINAL RESULTS COMPARISON")
print("="*70)

print(f"\n{'Method':<30} {'Accuracy':>10}")
print("-"*42)
print(f"{'ViT-only GP':<30} {accuracy_vit*100:>9.2f}%")
print(f"{'Handcrafted-only GP':<30} {accuracy_hc*100:>9.2f}%")
print(f"{'Fused Features GP':<30} {accuracy_fused*100:>9.2f}%")
print(f"{'Ensemble (Score Fusion)':<30} {accuracy_ensemble*100:>9.2f}%")
print("-"*42)

best_method = max([
    ('ViT-only GP', accuracy_vit),
    ('Handcrafted-only GP', accuracy_hc),
    ('Fused Features GP', accuracy_fused),
    ('Ensemble', accuracy_ensemble)
], key=lambda x: x[1])

print(f"\nBest: {best_method[0]} with {best_method[1]*100:.2f}%")

# Detailed report for best
print("\n" + "="*70)
print(f"CLASSIFICATION REPORT (Best: {best_method[0]})")
print("="*70)

if best_method[0] == 'ViT-only GP':
    best_preds = predictions_vit
elif best_method[0] == 'Handcrafted-only GP':
    best_preds = predictions_hc
elif best_method[0] == 'Fused Features GP':
    best_preds = predictions_fused
else:
    best_preds = predictions_ensemble

print(classification_report(test_lbl, best_preds,
                           target_names=[f"Class {i}" for i in range(num_classes)],
                           digits=4))

print("\n" + "="*70)
print("HANDCRAFTED FEATURES USED:")
for ft in hc_extractor.feature_types:
    print(f"  • {ft}")
print("="*70)

FUSED ViT + HANDCRAFTED FEATURES GP (FIXED)

Loading data...
Number of classes: 10
Training samples: 200 (20 per class)
Test samples: 2700

Loading ViT...

1. EXTRACTING ViT FEATURES


ViT features: 100%|██████████| 85/85 [00:41<00:00,  2.03it/s]


ViT features: Train (200, 768), Test (2700, 768)

2. EXTRACTING HANDCRAFTED FEATURES
Feature types: ['fourier', 'gabor', 'edge', 'lbp', 'hog', 'color_hist', 'wavelet', 'stats']


Extracting handcrafted features: 100%|██████████| 2700/2700 [15:09<00:00,  2.97it/s]


Handcrafted features: Train (200, 588), Test (2700, 588)

3. FUSING FEATURES (ViT + Handcrafted)
Combined features before PCA: (200, 1356)
Joint PCA: 128 components, 0.9411 variance
Fused features: Train (200, 128), Test (2700, 128)

4. TRAINING FUSED FEATURE GP

Training GP for class 0...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 1...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 2...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 3...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 4...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 5...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 6...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 7...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200



...


FUSED FEATURE GP ACCURACY: 0.1270 (12.70%)

5. VIT-ONLY GP (BASELINE)

Training GP for class 0...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 1...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 2...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 3...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 4...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 5...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 6...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 7...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 8...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 9...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200


Predicting: 100%|██████████| 10/10 [00:01<00:00,  6.39it/s]



VIT-ONLY GP ACCURACY: 0.1111 (11.11%)

6. HANDCRAFTED-ONLY GP

Training GP for class 0...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 1...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 2...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 3...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 4...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 5...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 6...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 7...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 8...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200

Training GP for class 9...
  Positive: 20, Negative: 180
  Final loss: 0.4324, Epochs: 200


Predicting: 100%|██████████| 10/10 [00:01<00:00,  6.65it/s]


HANDCRAFTED-ONLY GP ACCURACY: 0.0748 (7.48%)

7. ENSEMBLE (VIT + HANDCRAFTED SCORE FUSION)
ENSEMBLE GP ACCURACY: 0.1111 (11.11%)

FINAL RESULTS COMPARISON

Method                           Accuracy
------------------------------------------
ViT-only GP                        11.11%
Handcrafted-only GP                 7.48%
Fused Features GP                  12.70%
Ensemble (Score Fusion)            11.11%
------------------------------------------

Best: Fused Features GP with 12.70%

CLASSIFICATION REPORT (Best: Fused Features GP)
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000       300
     Class 1     1.0000    0.0133    0.0263       300
     Class 2     0.0000    0.0000    0.0000       300
     Class 3     0.0000    0.0000    0.0000       250
     Class 4     0.0957    0.9960    0.1747       250
     Class 5     0.0000    0.0000    0.0000       200
     Class 6     0.0000    0.0000    0.0000       250
     Class 7     0.8810    0